<!--
Copyright 2024 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# R&D Data Research Platform - Interactive Demo

This notebook demonstrates the full capabilities of the **R&D Data Research Platform** built on BigQuery.

## 🎯 Key Capabilities

1. **Generative AI** - LLM-powered summaries (Gemini 2.5 Pro)
2. **Semantic Search** - Vector similarity using 768-dimensional embeddings
3. **Graph Traversal** - Multi-hop pattern matching using GQL (GRAPH_TABLE)
4. **Relational SQL** - Traditional analytics and aggregations


## 🎯 Data Statistics

**8 Node Types:**
- 🧪 **Trial** - 1,200 clinical trials with 768-dim semantic embeddings
- 💊 **Drug** - Pharmaceutical compounds (UMLS CUIs)
- 🩺 **Disorder** - Medical conditions with SNOMED CT hierarchy + embeddings (1,657 disorders)
- 🔬 **MOA** - Mechanisms of Action (536 mechanisms)
- 📋 **TrialCriteria** - Requirements (17 criteria: disease indications, phase, status)
- 🏢 **Company** - Pharmaceutical manufacturers
- 🔢 **Phase** - Trial phases (9: Early Phase 1, Phase 1, 1/2, 2, 2/3, 3, 4, N/A)
- ✅ **Status** - Trial status (8: Recruiting, Active, Completed, Terminated, etc.)

**9 Edge Types:**
- Drug → Disorder (MayTreat): 8,405 relationships
- Disorder → Disorder (IsSubtypeOf): 1,701 SNOMED hierarchy relationships
- Trial → Phase (InPhase): 1,200 relationships
- Trial → Status (HasStatus): 1,200 relationships
- Trial → Criteria (Requires): 2,400 relationships
- Drug → MOA (HasMechanismOfAction)
- Drug → Company (ManufacturedBy): 2 relationships
- Trial → Drug (Uses)
- Trial → Disorder (Treats)

# Clinical Trials Analytics Demo
This notebook demonstrates an end-to-end clinical trials analytics workflow in BigQuery, leveraging unstructured data from GCS, autonomous embeddings, semantic search, and graph analysis.

In [ ]:
import subprocess
import sys
from IPython import get_ipython

# Install dependencies
packages = [
    "google-cloud-bigquery",
    "google-cloud-bigquery-storage",
    "google-cloud-storage",
    "pandas",
    "db-dtypes",
    "networkx",
    "matplotlib",
    "scipy",
    "pdf2image",
    "pypdf",
    "tqdm",
    "pyOpenSSL",
    "ipykernel",
    "bigquery-magics[spanner-graph-notebook]"
]

# Use extra-index-url to fallback to public PyPI for packages like scipy not found in internal index
subprocess.check_call([sys.executable, "-m", "pip", "install", "--extra-index-url", "https://pypi.org/simple/"] + packages)

# Load the extension
get_ipython().run_line_magic('load_ext', 'google.cloud.bigquery')

## Step 0: Documents in GCS
We list the files in the GCS bucket and preview the text content of the first document to understand the data we are working with.

In [ ]:
import base64
import io
import subprocess
import sys

from IPython.display import HTML, display

# 1. Install dependencies for visual rendering
try:
    subprocess.check_call(["apt-get", "install", "poppler-utils", "-y", "-q"])
except Exception as e:
    print(f"Could not install poppler-utils via apt-get: {e}")

subprocess.check_call([sys.executable, "-m", "pip", "install", "pdf2image"])

from pdf2image import convert_from_path

# 2. List and download a sample PDF from GCS
print("Listing files in GCS...")
try:
    files = (
        subprocess.check_output(
            ["gsutil", "ls", "gs://<PROJECT_ID>-clinical-trials-docs/*.pdf"]
        )
        .decode("utf-8")
        .splitlines()
    )

    if files:
        sample_pdf = files[0]
        print(f"Downloading sample PDF: {sample_pdf}")
        subprocess.check_call(["gsutil", "cp", sample_pdf, "sample.pdf"])

        # 3. Render 3 pages and display side-by-side
        print("Rendering first 3 pages...")
        images = convert_from_path("sample.pdf", first_page=1, last_page=3)

        # Convert images to base64 for HTML display
        html_content = "<div style='display: flex; justify-content: space-around; flex-wrap: wrap;'>"
        for i, img in enumerate(images):
            img.thumbnail((300, 400))  # Make smaller
            buffered = io.BytesIO()
            img.save(buffered, format="JPEG")
            img_str = base64.b64encode(buffered.getvalue()).decode()
            html_content += f"""
            <div style='margin: 10px; text-align: center;'>
                <p style='font-weight: bold;'>Page {i+1}</p>
                <img src='data:image/jpeg;base64,{img_str}' style='border: 1px solid #ccc; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);' />
            </div>
            """
        html_content += "</div>"

        display(HTML(html_content))
    else:
        print("No PDF files found in gs://<PROJECT_ID>-clinical-trials-docs/")
except Exception as e:
    print(f"Error exploring PDFs: {e}")

## Step 1: Generate Object Table
We create an external object table in BigQuery to reference the raw documents stored in GCS (`gs://<PROJECT_ID>-clinical-trials-docs/`).

In [ ]:
# sql_engine: bigquery
# output_variable: object_table
# start _sql
_sql = """
CREATE EXTERNAL TABLE IF NOT EXISTS `<PROJECT_ID>.<DATASET_ID>.object_table`
WITH CONNECTION `us.cloud_ai_resources`
OPTIONS (
  object_metadata = 'DIRECTORY',
  uris = ['gs://<PROJECT_ID>-clinical-trials-docs/*']
  );
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

object_table = _bqsqlcell.run(_sql)
object_table

##Step 2: Define Full Schema with Autonomous Embeddings
We define the full table schema with all required columns and a stored generated embedding column for `StudyTitle`. This table will be populated in the next step.

In [ ]:
# sql_engine: bigquery
# output_variable: ClinicalTrialMasterData_embedded
# start _sql
_sql = """
CREATE TABLE IF NOT EXISTS `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded` (
  uri STRING,
  Sponsor STRING,
  StudyTitle STRING,
  PreferredUMLSName ARRAY<STRING>,
  NCT_Number STRING,
  Phase STRING,
  Trial_Status STRING,
  Disease_Areas STRING,
  Targeted_Enrollment INT64,
  Company STRING,
  semantic_text STRING,
  name STRING,
  preferred_name STRING,
  semantic_type ARRAY<STRING>,
  definition STRING,
  mesh_code STRING,
  mesh_codes ARRAY<STRING>,
  hpo_codes ARRAY<STRING>,
  snomed_id STRING,
  snomed_hierarchy ARRAY<STRING>,
  drug_name STRING,
  atc_code STRING,
  atc_codes ARRAY<STRING>,
  rxnorm_code STRING,
  trade_names ARRAY<STRING>,
  ema_url ARRAY<STRING>,
  source_level INT64,
  drug_preferred_name STRING,
  drug_semantic_type ARRAY<STRING>,
  criteria_type STRING,
  criteria_text STRING,
  phase_id STRING,
  status_name STRING,
  status_description STRING,
  objectRef STRING,
  ClinicalTrialMasterData_embedded STRUCT<result ARRAY<FLOAT64>, status STRING>
    GENERATED ALWAYS AS (AI.EMBED(
      StudyTitle,
      connection_id => 'us.cloud_ai_resources',
      endpoint => 'text-embedding-005'
    ))
    STORED
    OPTIONS( asynchronous = TRUE )
);
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

ClinicalTrialMasterData_embedded = _bqsqlcell.run(_sql)
ClinicalTrialMasterData_embedded

## Step 3: Populate Full Table using AI.GENERATE
We use `AI.GENERATE` to extract all the columns required for the full schema and insert them into the table created in Step 2.

In [ ]:
# sql_engine: bigquery
# output_variable: ClinicalTrialMasterData_embedded
# start _sql
_sql = """
-- Check if table is empty before inserting
IF NOT EXISTS (SELECT 1 FROM `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded` LIMIT 1) THEN
  INSERT INTO `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`
  (uri, Sponsor, StudyTitle, PreferredUMLSName, NCT_Number, Phase, Trial_Status, Disease_Areas, Targeted_Enrollment, Company, semantic_text, name, preferred_name, semantic_type, definition, mesh_code, mesh_codes, hpo_codes, snomed_id, snomed_hierarchy, drug_name, atc_code, atc_codes, rxnorm_code, trade_names, ema_url, source_level, drug_preferred_name, drug_semantic_type, criteria_type, criteria_text, phase_id, status_name, status_description, objectRef)
  SELECT
    uri, Sponsor, StudyTitle, PreferredUMLSName, NCT_Number, Phase, Trial_Status, Disease_Areas, Targeted_Enrollment, Company, semantic_text, name, preferred_name, semantic_type, definition, mesh_code, mesh_codes, hpo_codes, snomed_id, snomed_hierarchy, drug_name, atc_code, atc_codes, rxnorm_code, trade_names, ema_url, source_level, drug_preferred_name, drug_semantic_type, criteria_type, criteria_text, phase_id, status_name, status_description,
    uri AS objectRef
  FROM (
    SELECT
      uri,
      AI.GENERATE(
        prompt => STRUCT('Extract the following fields from the clinical trial document: Sponsor, StudyTitle, PreferredUMLSName (as array), NCT_Number, Phase, Trial_Status, Disease_Areas, Targeted_Enrollment (as integer), Company, semantic_text, name, preferred_name, semantic_type (as array), definition, mesh_code, mesh_codes (as array), hpo_codes (as array), snomed_id, snomed_hierarchy (as array), drug_name, atc_code, atc_codes (as array), rxnorm_code, trade_names (as array), ema_url (as array), source_level (as integer), drug_preferred_name, drug_semantic_type (as array), criteria_type, criteria_text, phase_id, status_name, status_description.', OBJ.GET_ACCESS_URL(OBJ.MAKE_REF(uri, 'us.cloud_ai_resources'), 'r')),
        endpoint => 'gemini-2.5-flash',
        output_schema => 'Sponsor STRING, StudyTitle STRING, PreferredUMLSName ARRAY<STRING>, NCT_Number STRING, Phase STRING, Trial_Status STRING, Disease_Areas STRING, Targeted_Enrollment INT64, Company STRING, semantic_text STRING, name STRING, preferred_name STRING, semantic_type ARRAY<STRING>, definition STRING, mesh_code STRING, mesh_codes ARRAY<STRING>, hpo_codes ARRAY<STRING>, snomed_id STRING, snomed_hierarchy ARRAY<STRING>, drug_name STRING, atc_code STRING, atc_codes ARRAY<STRING>, rxnorm_code STRING, trade_names ARRAY<STRING>, ema_url ARRAY<STRING>, source_level INT64, drug_preferred_name STRING, drug_semantic_type ARRAY<STRING>, criteria_type STRING, criteria_text STRING, phase_id STRING, status_name STRING, status_description STRING',
        connection_id => '<PROJECT_ID>.us.cloud_ai_resources'
      ).* EXCEPT (full_response, status)
    FROM
      `<PROJECT_ID>.<DATASET_ID>.object_table`
  );
  SELECT 'Data inserted successfully.' AS status;
ELSE
  SELECT 'Table is not empty. Skipping insertion.' AS status;
END IF;
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

ClinicalTrialMasterData_embedded = _bqsqlcell.run(_sql)
ClinicalTrialMasterData_embedded

Previewing the tables



In [ ]:
%%bigquery --project <PROJECT_ID>
-- Preview the populated table
SELECT * FROM `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded` LIMIT 5;

## Step 4: [Optional] Parse Documents using AI.PARSE_DOCUMENT
We use `AI.PARSE_DOCUMENT` to extract content from documents in GCS. This step is optional and demonstrates document parsing capabilities. This is "nifty" because it can handle complex layouts, tables, and formatting much better than traditional OCR, and it returns structured chunks.

In [ ]:
# sql_engine: bigquery
# output_variable: chunks
# start _sql
_sql = """
SELECT
  uri,
  chunk_id,
  content
FROM AI.parse_document(
  (SELECT * FROM `<PROJECT_ID>.<DATASET_ID>.object_table` LIMIT 5),
  endpoint => 'gemini-2.5-flash'
)
LIMIT 5;
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

chunks = _bqsqlcell.run(_sql)
chunks

## Step 5: AI.Search (Semantic and Hybrid)
We perform semantic search using the native `AI.SEARCH` function in BigQuery, leveraging the embeddings generated in previous steps.

In [ ]:
# sql_engine: bigquery
# output_variable: top_5_cancer
# start _sql
_sql = """
SELECT
  base.StudyTitle,
  distance
FROM AI.SEARCH(
  TABLE `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`,
  'StudyTitle',
  'Cancer treated by MK-3475',
  top_k => 10
)
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

top_5_cancer = _bqsqlcell.run(_sql)
top_5_cancer

### Advanced Hybrid Search: Native HYBRID Mode
In this example, we show how combining lexical (keyword) search with semantic search can improve results using the native `HYBRID` mode in `AI.SEARCH`. Pure semantic search might return documents that are conceptually similar but miss specific keyword matches (like specific drug names or trial IDs) that are critical.

Here we search for trials related to "immunotherapy" but also want to ensure we match the specific drug "Keytruda". We pass both terms in the query and set `mode => 'HYBRID'`. This combines the power of vector search with keyword matching without needing complex regex filters.

In [ ]:
# sql_engine: bigquery
# output_variable: hybrid
# start _sql
_sql = """
SELECT
  base.StudyTitle
FROM AI.SEARCH(
  TABLE `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`,
  'StudyTitle',
  'Cancer treated by MK-3475',
  mode => 'hybrid',
  top_k => 10
)
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

hybrid = _bqsqlcell.run(_sql)
hybrid

## Step 6: Graph Generation and Traversal
We generate a property graph on top of our table and perform traversal queries to find relationships between trials.

In [ ]:
# sql_engine: bigquery
# output_variable: graph_views
# start _sql
_sql = """
-- Create views for expanded graph nodes and edges focused on Trials, Drugs, and Sponsors

-- 1. Drug Nodes
CREATE OR REPLACE VIEW `<PROJECT_ID>.<DATASET_ID>.drug_nodes` AS
SELECT DISTINCT drug_name AS drug_name
FROM `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`
WHERE drug_name IS NOT NULL;

-- 2. Trial -> Drug Edges
CREATE OR REPLACE VIEW `<PROJECT_ID>.<DATASET_ID>.trial_drug_edges` AS
SELECT DISTINCT NCT_Number AS trial_id, drug_name AS drug_name
FROM `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`
WHERE drug_name IS NOT NULL AND NCT_Number IS NOT NULL;

-- 3. Sponsor Nodes
CREATE OR REPLACE VIEW `<PROJECT_ID>.<DATASET_ID>.sponsor_nodes` AS
SELECT DISTINCT Sponsor AS sponsor_name
FROM `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`
WHERE Sponsor IS NOT NULL;

-- 4. Trial -> Sponsor Edges
CREATE OR REPLACE VIEW `<PROJECT_ID>.<DATASET_ID>.trial_sponsor_edges` AS
SELECT DISTINCT NCT_Number AS trial_id, Sponsor AS sponsor_name
FROM `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`
WHERE Sponsor IS NOT NULL AND NCT_Number IS NOT NULL;
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

graph_views = _bqsqlcell.run(_sql)
graph_views

In [ ]:
# sql_engine: bigquery
# output_variable: graph_gen
# start _sql
_sql = """
-- Create Property Graph focused on Trials, Drugs, and Sponsors
CREATE OR REPLACE PROPERTY GRAPH `<PROJECT_ID>.<DATASET_ID>.clinical_trial_graph`
NODE TABLES (
  `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`
    KEY (NCT_Number)
    LABEL Trial
    PROPERTIES (NCT_Number, StudyTitle, Disease_Areas, Phase),

  `<PROJECT_ID>.<DATASET_ID>.drug_nodes`
    KEY (drug_name)
    LABEL Drug
    PROPERTIES (drug_name),

  `<PROJECT_ID>.<DATASET_ID>.sponsor_nodes`
    KEY (sponsor_name)
    LABEL Sponsor
    PROPERTIES (sponsor_name)
)
EDGE TABLES (
  `<PROJECT_ID>.<DATASET_ID>.trial_drug_edges`
    KEY (trial_id, drug_name)
    SOURCE KEY (trial_id) REFERENCES `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded` (NCT_Number)
    DESTINATION KEY (drug_name) REFERENCES `<PROJECT_ID>.<DATASET_ID>.drug_nodes` (drug_name)
    LABEL TestsDrug,

  `<PROJECT_ID>.<DATASET_ID>.trial_sponsor_edges`
    KEY (trial_id, sponsor_name)
    SOURCE KEY (trial_id) REFERENCES `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded` (NCT_Number)
    DESTINATION KEY (sponsor_name) REFERENCES `<PROJECT_ID>.<DATASET_ID>.sponsor_nodes` (sponsor_name)
    LABEL SponsoredBy
);
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

graph_gen = _bqsqlcell.run(_sql)
graph_gen

In [ ]:
# sql_engine: bigquery
# output_variable: graph_preview
# start _sql
_sql = """
SELECT * FROM GRAPH_TABLE(
  `<PROJECT_ID>.<DATASET_ID>.clinical_trial_graph`
  MATCH (t:Trial)-[:SponsoredBy]->(s:Sponsor)
  RETURN t.StudyTitle AS source, s.sponsor_name AS target, 'SponsoredBy' AS rel
);
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

graph_preview = _bqsqlcell.run(_sql)
graph_preview

## Step 7: Visualize the Graph
We use BigQuery's native graph visualization capabilities to render the graph directly in the notebook. This requires `bigquery_magics` to be installed and the query to return graph elements in JSON format using `TO_JSON()`.

In [ ]:
%%bigquery --graph --project <PROJECT_ID>
GRAPH `<PROJECT_ID>.<DATASET_ID>.clinical_trial_graph`
MATCH p = (d:Drug)<-[:TestsDrug]-(t:Trial)-[:SponsoredBy]->(s:Sponsor)
RETURN TO_JSON(p) AS path
LIMIT 100;

## Step 8: Advanced Graph Traversal
Graph traversal is useful in clinical trials for several scenarios, such as identifying trials with shared sponsors or finding multi-hop connections between entities.

-- Combined Search + Graph Traversal Use Case
-- 1. Find trials about 'Metabolic diseases' using Semantic Search (Hybrid mode)
-- 2. Traverse the graph to find other trials testing the same drugs
-- 3. Filter for Phase 3 source trials
-- 4. Summarize the connection using AI.GENERATE

In [ ]:
# sql_engine: bigquery
# output_variable: df
# start _sql
_sql = """
WITH relevant_trials AS (
  SELECT base.NCT_Number
  FROM AI.SEARCH(
    TABLE `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`,
    'StudyTitle',
    'Metabolic diseases',
    mode => 'hybrid',
    top_k => 10
  )
),
graph_data AS (
  SELECT * FROM GRAPH_TABLE(
    `<PROJECT_ID>.<DATASET_ID>.clinical_trial_graph`
    MATCH (t:Trial)-[:TestsDrug]->(d:Drug)<-[:TestsDrug]-(other:Trial)
    WHERE t.Phase = 'Phase 3'
    RETURN t.NCT_Number AS t_nct, t.StudyTitle AS source_trial, d.drug_name AS drug, other.StudyTitle AS related_trial, other.NCT_Number AS other_nct, other.Disease_Areas AS related_disease
  )
),
raw_results AS (
  SELECT DISTINCT
    gd.source_trial,
    gd.drug,
    gd.related_trial,
    gd.related_disease
  FROM graph_data gd
  JOIN relevant_trials rt ON gd.t_nct = rt.NCT_Number
  WHERE gd.t_nct != gd.other_nct
  QUALIFY ROW_NUMBER() OVER(PARTITION BY gd.drug ORDER BY gd.related_trial) <= 2
)
SELECT
  source_trial,
  drug,
  related_trial,
  related_disease,
  AI.GENERATE(
    prompt => 'Analyze these two clinical trials testing the same drug (' || drug || '). Provide a pithy, one-sentence cross-indication laymans insight starting with "Based on the findings of these two studies...". Trial 1: ' || source_trial || ' | Trial 2: ' || related_trial,
    endpoint => 'gemini-2.5-flash',
    output_schema => 'summary STRING',
    connection_id => '<PROJECT_ID>.us.cloud_ai_resources'
  ).summary AS cross_trial_insight
FROM raw_results
LIMIT 5;
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

df = _bqsqlcell.run(_sql)
df

In [ ]:
from google.api_core.client_info import ClientInfo
from google.cloud import bigquery

if 'df_combined' in globals():
    client_info = ClientInfo(user_agent="cloud-solutions/data-to-ai-nb-v2")
        client = bigquery.Client(project='<PROJECT_ID>', client_info=client_info)
    insights_text = " ".join(df_combined['cross_trial_insight'].tolist())

    query = """
    SELECT AI.GENERATE(
      prompt => 'Summarize these clinical trial insights in simple, layman terms for a non-medical audience in exactly one sentence. Insights: ' || @insights,
      endpoint => 'gemini-2.5-flash',
      output_schema => 'summary STRING',
      connection_id => '<PROJECT_ID>.us.cloud_ai_resources'
    ).summary AS layman_summary
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("insights", "STRING", insights_text),
        ]
    )

    df_layman = client.query(query, job_config=job_config).to_dataframe()

    print("
Layman's 'So What' Summary:")
    print(df_layman['layman_summary'][0])

## Step 9: Scale - Creating a Vector Index
To scale semantic search to millions of documents, we can create a vector index on the embedding column. BigQuery supports IVF (Inverted File) index type for efficient approximate nearest neighbor search.

In [ ]:
# sql_engine: bigquery
# output_variable: index_creation
# start _sql
_sql = """
-- Create Vector Index (IVF Type)
-- NOTE: This cell is commented out because BigQuery requires a minimum of 5000 rows
-- to create an IVF vector index. The current table has ~1205 rows.
-- For small datasets, semantic search via AI.SEARCH works perfectly without an index.
-- Uncomment this when your dataset grows beyond 5000 rows.

/*
CREATE OR REPLACE VECTOR INDEX `<PROJECT_ID>.<DATASET_ID>.clinical_trial_index`
ON `<PROJECT_ID>.<DATASET_ID>.ClinicalTrialMasterData_embedded`(ClinicalTrialMasterData_embedded)
OPTIONS(
  distance_type = 'COSINE',
  index_type = 'IVF'
);
*/
SELECT 'Skipping index creation: Table has fewer than 5000 rows.' AS status;
"""  # end _sql
from google.colab.sql import bigquery as _bqsqlcell

index_creation = _bqsqlcell.run(_sql)
index_creation